# TED Talks Transcript Extractor

Ze slugów znalezionych przez finder `ted_candidates.csv` wyciąga tytuł, opis i transkrypcję w każdym z docelowych języków (format SRT i czysty tekst)

#### 1. Zależności

In [1]:
%pip install --quiet requests beautifulsoup4 pandas lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1
[notice] To update, run: C:\Users\estep\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import json
import re
import time
from typing import Optional

import pandas as pd
import requests
from bs4 import BeautifulSoup

#### 2. Ustawienia

In [ ]:
TARGET_LANGUAGES = ["zh", "en", "fi", "fr", "he", "ja", "pl"]
CHINESE_FALLBACKS = ["zh", "zh-cn", "zh-tw"]

CANDIDATES_PATH = "LLMs/data/ted_candidates.csv"
OUTPUT_PATH = "LLMs/data/ted_talks_transcripts.tsv"

POLITE_DELAY = 0.5

GRAPHQL_URL = "https://www.ted.com/graphql"

UA = (
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36"
)

HEADERS = {
    "User-Agent": UA,
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

GRAPHQL_HEADERS = {
    "User-Agent": UA,
    "Accept": "application/json",
    "Content-Type": "application/json",
    "Origin": "https://www.ted.com",
    "Referer": "https://www.ted.com/",
}

session = requests.Session()
session.headers.update(HEADERS)

#### 3. Pobieranie

Wczytanie strony talka i odczyt osadzone JSON `__NEXT_DATA__` w celu wyciągnięcia video_id, tytułu i opisu

In [ ]:
def fetch_talk_metadata(slug: str) -> dict:
    """Pobierz video_id, tytuł i opis ze strony prelekcji TED."""
    url = f"https://www.ted.com/talks/{slug}"
    resp = session.get(url, timeout=30)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "lxml")
    tag = soup.find("script", id="__NEXT_DATA__")
    if tag is None or not tag.string:
        raise RuntimeError(f"__NEXT_DATA__ not found on {url}")

    data = json.loads(tag.string)
    video_data = data.get("props", {}).get("pageProps", {}).get("videoData") or {}

    return {
        "video_id": str(video_data.get("id") or ""),
        "title": video_data.get("title") or "",
        "description": video_data.get("description") or "",
    }

In [5]:
slugs = pd.read_csv(CANDIDATES_PATH)["slug"].tolist()
print(f"Wczytano {len(slugs)} slugow z {CANDIDATES_PATH}")

Wczytano 282 slugow z ted_candidates.csv


#### 4. Ekstrakcja

Budowanie zapytania GraphQL i formatowanie znaczników do SRT. Pętla przechodzi przez wszystkie slugi i wyciąga transkrypcje

In [ ]:
def _build_query(video_id: str, language: str) -> str:
    return (
        '{ translation(videoId:"' + video_id + '", language:"' + language + '") '
        '{ paragraphs { cues { text time endTime } } } }'
    )


def _format_srt_time(ms: int) -> str:
    hours = ms // 3_600_000
    minutes = (ms % 3_600_000) // 60_000
    seconds = (ms % 60_000) // 1000
    millis = ms % 1000
    return f"{hours:02d}:{minutes:02d}:{seconds:02d},{millis:03d}"


NL = chr(10)


def _graphql_transcript(video_id: str, language: str, timeout: int = 30) -> Optional[dict]:
    """Pobiera transkrypcję z GraphQL TED-a, zwraca dict {plain, timed (SRT)} lub None."""
    payload = {"operationName": None, "query": _build_query(video_id, language)}
    resp = session.post(
        GRAPHQL_URL,
        json=payload,
        headers=GRAPHQL_HEADERS,
        timeout=timeout,
    )
    if resp.status_code != 200:
        print(f"  [{language}] HTTP {resp.status_code}")
        return None

    try:
        body = resp.json()
    except ValueError:
        print(f"  [{language}] invalid JSON response")
        return None

    translation = (body.get("data") or {}).get("translation")
    if not translation:
        return None

    plain_lines = []
    srt_blocks = []
    cue_idx = 0

    for paragraph in translation.get("paragraphs") or []:
        cues = paragraph.get("cues") or []
        plain_parts = []
        for c in cues:
            text = (c.get("text") or "").strip()
            if not text:
                continue
            text = re.sub(r"\s+", " ", text)
            plain_parts.append(text)

            ts_start = c.get("time")
            ts_end = c.get("endTime")
            if ts_start is not None and ts_end is not None:
                cue_idx += 1
                time_line = f"{_format_srt_time(ts_start)} --> {_format_srt_time(ts_end)}"
                block = f"{cue_idx}{NL}{time_line}{NL}{text}"
                srt_blocks.append(block)
        plain_text = " ".join(plain_parts).strip()
        if plain_text:
            plain_lines.append(plain_text)

    if not plain_lines:
        return None
    return {
        "plain": NL.join(plain_lines),
        "timed": (NL * 2).join(srt_blocks),
    }


def fetch_transcript(video_id: str, language: str, polite_delay: float = POLITE_DELAY) -> Optional[dict]:
    """Próbuje kolejnych kodów języka (zh -> zh, zh-cn, zh-tw); zwraca pierwsze trafienie."""
    candidates = CHINESE_FALLBACKS if language == "zh" else [language]
    for lang_code in candidates:
        result = _graphql_transcript(video_id, lang_code)
        if result:
            time.sleep(polite_delay)
            return result
        time.sleep(polite_delay)
    return None

In [7]:
rows = []
total = len(slugs)

for i, slug in enumerate(slugs, 1):
    print(f"[{i}/{total}] {slug}")
    try:
        meta = fetch_talk_metadata(slug)
    except Exception as exc:
        print(f"  blad metadanych: {exc}")
        continue

    transcripts = {}
    for lang in TARGET_LANGUAGES:
        result = fetch_transcript(meta["video_id"], lang)
        if result:
            transcripts[lang] = result["plain"]
            transcripts[f"{lang}_timed"] = result["timed"]
            status = f"{len(result['plain']):,} chars"
        else:
            transcripts[lang] = None
            transcripts[f"{lang}_timed"] = None
            status = "brak"
        print(f"  {lang}: {status}")

    rows.append({
        "slug": slug,
        "Ted talk title": meta["title"],
        "description": meta["description"],
        **transcripts,
    })

print(f"Zebrano {len(rows)} wierszy")

[1/282] adam_grant_how_to_stop_languishing_and_start_finding_flow
  zh: 5,113 chars
  en: 13,505 chars
  fi: 12,389 chars
  fr: 14,210 chars
  he: 10,817 chars
  ja: 6,358 chars
  pl: 12,911 chars
[2/282] molly_wright_how_every_child_can_thrive_by_five
  zh: 1,569 chars
  en: 4,231 chars
  fi: 4,297 chars
  fr: 4,930 chars
  he: 3,516 chars
  ja: 2,098 chars
  pl: 4,399 chars
[3/282] andrea_berchowitz_the_link_between_menopause_and_gender_inequity_at_work
  zh: 2,425 chars
  en: 7,804 chars
  fi: 7,112 chars
  fr: 8,829 chars
  he: 6,303 chars
  ja: 3,528 chars
  pl: 7,693 chars
[4/282] wendy_de_la_rosa_why_talking_to_your_friends_can_help_you_save_money
  zh: 1,096 chars
  en: 3,609 chars
  fi: 3,427 chars
  fr: 3,836 chars
  he: 2,864 chars
  ja: 1,463 chars
  pl: 3,300 chars
[5/282] elif_shafak_if_trees_could_speak
  zh: 1,014 chars
  en: 2,678 chars
  fi: 2,634 chars
  fr: 3,147 chars
  he: 2,136 chars
  ja: 1,050 chars
  pl: 2,609 chars
[6/282] ariel_waldman_the_invisible_life_hid

### 5. Zapis

Porządkowanie, wstępna weryfikacja i zapis do pliku

In [12]:
df = pd.DataFrame(rows)

core_cols = ["slug", "Ted talk title", "description"]
for lang in TARGET_LANGUAGES:
    core_cols += [lang, f"{lang}_timed"]

extra_cols = [c for c in df.columns if c not in core_cols]
df = df[core_cols + extra_cols]

In [13]:
df.head(3)

,slug,Ted talk title,description,zh,zh_timed,en,en_timed,fi,fi_timed,fr,fr_timed,he,he_timed,ja,ja_timed,pl,pl_timed
0,adam_grant_how_to_stop_languishing_and_start_f...,How to stop languishing and start finding flow,"Have you found yourself staying up late, joyle...",我知道大家都有 长长的待办事项列表， 但是我讨厌浪费时间做这些 所以我有个“不办”事项列表。...,"1\n00:00:00,413 --> 00:00:02,707\n我知道大家都有 长长的待...","I know you all have long to-do lists, but I ha...","1\n00:00:00,413 --> 00:00:02,707\nI know you a...","Teillä on pitkiä to do -listoja, mutta vihaan ...","1\n00:00:00,413 --> 00:00:02,707\nTeillä on pi...",Vous avez tous de longues listes de choses à f...,"1\n00:00:00,288 --> 00:00:02,749\nVous avez to...","אני יודע שלכולכם יש רשימות ארוכות של מטלות, אב...","1\n00:00:00,000 --> 00:00:02,290\nאני יודע שלכ...",みんな長い「やることリスト」を 持っていると思いますが 私は時間を無駄にしたくないので 「や...,"1\n00:00:00,413 --> 00:00:02,790\nみんな長い「やることリス...","Wiem, że wszyscy mają listy rzeczy do zrobieni...","1\n00:00:00,413 --> 00:00:02,749\nWiem, że wsz..."
1,molly_wright_how_every_child_can_thrive_by_five,How every child can thrive by five,"""What if I was to tell you that a game of peek...",（小朋友笑聲）\n如果我同你講 伏匿匿呢個遊戲可以改變世界 你係唔係覺得好荒謬？ 我今日就係...,"1\n00:00:02,796 --> 00:00:04,556\n（小朋友笑聲）\n\n2...",[Baby cooing]\nWhat if I was to tell you that ...,"1\n00:00:02,796 --> 00:00:04,556\n[Baby cooing...","[Vauva jokeltaa]\nMitä jos kertoisin teille, e...","1\n00:00:02,796 --> 00:00:04,556\n[Vauva jokel...",[Gazouilli d’un bébé]\nEt si je vous disais qu...,"1\n00:00:02,796 --> 00:00:04,556\n[Gazouilli d...",[תינוק ממלמל]\nמה אם הייתי אומרת לכם שמשחק “קו...,"1\n00:00:02,796 --> 00:00:04,556\n[תינוק ממלמל...",[赤ちゃんの声]\nもし「いないいないばぁ」で 世界を変えられると言われたら どう思いますか...,"1\n00:00:02,796 --> 00:00:04,556\n[赤ちゃんの声]\n\n...",(Gaworzenie dziecka)\nA gdybym wam powiedziała...,"1\n00:00:02,636 --> 00:00:04,396\n(Gaworzenie ..."
2,andrea_berchowitz_the_link_between_menopause_a...,The link between menopause and gender inequity...,"Hot flashes, joint pain, anxiety, depression, ...",在全美五百强公司里，\n只有 42 位女性CEO。 在别的国家，数据也是类似的， 有的地方还...,"1\n00:00:01,873 --> 00:00:04,375\n在全美五百强公司里，\n...","Of America's 500 largest companies, only 42 ha...","1\n00:00:01,873 --> 00:00:07,045\nOf America's...",Amerikan 500 suurimmasta yhtiöstä 42:ssa on na...,"1\n00:00:01,873 --> 00:00:07,045\nAmerikan 500...",Seules 42 des 500 plus grandes entreprises des...,"1\n00:00:01,873 --> 00:00:07,045\nSeules 42 de...","באמריקה, מתוך 500 החברות הגדולות, רק ל-41 יש מ...","1\n00:00:01,831 --> 00:00:06,627\nבאמריקה, מתו...",アメリカの大企業500社の内 女性CEOはたったの42人しかいません 他の国に目を向けると ...,"1\n00:00:01,873 --> 00:00:07,045\nアメリカの大企業500社...","W 500 największych amerykańskich firmach, zale...","1\n00:00:01,831 --> 00:00:06,961\nW 500 najwię..."


In [10]:
print(df["en_timed"].iloc[1])

1
00:00:02,796 --> 00:00:04,556
[Baby cooing]

2
00:00:20,076 --> 00:00:22,796
What if I was to tell you

3
00:00:22,836 --> 00:00:27,756
that a game of peekaboo could change the world?

4
00:00:29,076 --> 00:00:31,356
Sounds impossible, right?

5
00:00:31,876 --> 00:00:35,756
Well, I’m here today to prove it’s not.

6
00:00:36,436 --> 00:00:38,116
Hi, I’m Molly and I’m seven.

7
00:00:38,156 --> 00:00:40,436
And this is my little friend, Ari.

8
00:00:40,636 --> 00:00:42,316
Say “Hi,” Ari.

9
00:00:43,276 --> 00:00:44,596
Hi.

10
00:00:45,116 --> 00:00:48,156
Oh, and this is my neighbor, Amarjot.

11
00:00:48,756 --> 00:00:53,036
He has to take Ari away now to get ready for our experiment.

12
00:00:53,156 --> 00:00:55,516
But don't worry, they'll be back.

13
00:00:56,916 --> 00:01:00,516
My talk today is about some powerful things you grownups can do.

14
00:01:00,556 --> 00:01:03,756
that shape us as children and the adults we become.

15
00:01:04,556 --> 00:01:06,116
How do I know

In [3]:
df.to_csv(OUTPUT_PATH, sep="\t", index=False)
print(f"Saved {len(df)} row(s) -> {OUTPUT_PATH}")

NameError: name 'df' is not defined

In [ ]:
df = pd.read_csv(OUTPUT_PATH, sep="\t")

In [7]:
print(df["pl_timed"].iloc[0])

1
00:00:00,413 --> 00:00:02,749
Wiem, że wszyscy mają listy rzeczy do zrobienia,

2
00:00:02,790 --> 00:00:07,503
ale sam nie znoszę marnowania czasu, więc mam listy rzeczy do nierobienia.

3
00:00:07,545 --> 00:00:09,547
Nie przeglądaj mediów społecznościowych,

4
00:00:09,589 --> 00:00:11,257
nie sprawdzaj telefonu w łóżku

5
00:00:11,299 --> 00:00:12,800
i nie włączaj telewizora,

6
00:00:12,842 --> 00:00:15,344
jeśli nie wiesz, co chcesz obejrzeć.

7
00:00:15,720 --> 00:00:19,057
W zeszłym roku łamałem wszystkie te zasady.

8
00:00:19,098 --> 00:00:21,059
Kładłem się grubo po północy,

9
00:00:21,059 --> 00:00:22,518
tonąłem w negatywnych newsach,

10
00:00:22,518 --> 00:00:24,979
ciągle grałem w Scrabble online

11
00:00:24,979 --> 00:00:27,398
i kompulsywnie oglądałem całe sezony seriali,

12
00:00:27,398 --> 00:00:29,776
które nawet nie były dobre.

13
00:00:30,777 --> 00:00:32,653
Rano budziłem się nieprzytomny

14
00:00:32,653 --> 00:00:35,490
i obiecywałem sobie, że tym razem